In [1]:
import os, json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

## The schema stub

In [2]:
def get_text_edit_schema(model):
    if model.startswith("claude-3"):
        return {"type": "text_editor_20250124", "name": "str_replace_editor"}
    else:
        return {"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"}

text_edit_schema = get_text_edit_schema(model)
print(text_edit_schema)

{'type': 'text_editor_20250728', 'name': 'str_replace_based_edit_tool'}


## Your implementation of the commands

Claude only asks for operations. These functions do them, restricted to a sandbox folder so Claude cannot touch anything else.

In [3]:
SANDBOX = os.path.abspath("sandbox")
os.makedirs(SANDBOX, exist_ok=True)

def safe_path(path):
    full = os.path.abspath(os.path.join(SANDBOX, path.lstrip("/\\")))
    if not full.startswith(SANDBOX):
        raise ValueError("Path outside sandbox")
    return full

def view(path, view_range=None):
    full = safe_path(path)
    if os.path.isdir(full):
        return "\n".join(os.listdir(full)) or "(empty directory)"
    with open(full, encoding="utf-8") as f:
        lines = f.read().splitlines()
    start, end = (view_range or [1, len(lines)])
    end = len(lines) if end == -1 else end
    return "\n".join(f"{i}: {l}" for i, l in enumerate(lines[start-1:end], start))

def create(path, file_text):
    full = safe_path(path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w", encoding="utf-8") as f:
        f.write(file_text)
    return f"Created {path}"

def str_replace(path, old_str, new_str):
    full = safe_path(path)
    with open(full, encoding="utf-8") as f:
        text = f.read()
    if text.count(old_str) != 1:
        raise ValueError(f"old_str must match exactly once, found {text.count(old_str)}")
    with open(full, "w", encoding="utf-8") as f:
        f.write(text.replace(old_str, new_str))
    return "Replaced text"

def insert(path, insert_line, insert_text):
    full = safe_path(path)
    with open(full, encoding="utf-8") as f:
        lines = f.read().splitlines()
    lines.insert(insert_line, insert_text)
    with open(full, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return f"Inserted at line {insert_line}"

def run_text_edit(tool_input):
    cmd = tool_input["command"]
    if cmd == "view":
        return view(tool_input["path"], tool_input.get("view_range"))
    if cmd == "create":
        return create(tool_input["path"], tool_input["file_text"])
    if cmd == "str_replace":
        return str_replace(tool_input["path"], tool_input["old_str"], tool_input["new_str"])
    if cmd == "insert":
        return insert(tool_input["path"], tool_input["insert_line"], tool_input["insert_text"])
    raise ValueError(f"Unsupported command: {cmd}")

## Conversation loop

In [4]:
def run_conversation(messages, max_turns=10):
    for _ in range(max_turns):
        response = client.messages.create(
            model=model, max_tokens=2000, messages=messages, tools=[text_edit_schema]
        )
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            return response

        results = []
        for block in response.content:
            if block.type == "tool_use":
                try:
                    out = run_text_edit(block.input)
                    results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(out)})
                except Exception as e:
                    results.append({"type": "tool_result", "tool_use_id": block.id,
                                    "content": f"Error: {e}", "is_error": True})
        messages.append({"role": "user", "content": results})

messages = [{"role": "user", "content":
    "Create a file called hello.py that prints a greeting, then change the greeting to Spanish."}]
final = run_conversation(messages)
print("\n".join(b.text for b in final.content if b.type == "text"))

Perfect! I've created the file `hello.py` with a greeting and then changed it to Spanish. The file now contains:

```python
print("¡Hola, Mundo!")
```

When run, this will print the Spanish greeting "¡Hola, Mundo!" (Hello, World!).


## Check the result

In [5]:
print(open("sandbox/hello.py").read())

print("Â¡Hola, Mundo!")

